### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

In [2]:
import requests
import json
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)



ROOT: /home/flavio/uv/lca-lc-foundations
SRC added: /home/flavio/uv/lca-lc-foundations/src


### Defaults

In [ ]:
url_gdc_cases = "https://api.gdc.cancer.gov/cases"
url_gdc_files = "https://api.gdc.cancer.gov/files"
url_gdc_data  = f"https://api.gdc.cancer.gov/data/%s"

root_data = "/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga"

case = "Breast Cancer"
dir_case = case.lower().replace(' ', '_')

root_case = os.path.join(root_data, dir_case)

root_case, os.listdir(root_case)

In [ ]:
def get_case_uuid(barcode: str):
    
    try:
        filters = {
            "op": "in",
            "content": {
                "field": "submitter_id",
                "value": [barcode]
            }
        }

        params = {
            "filters": json.dumps(filters),
            "fields": "case_id,submitter_id",
            "format": "JSON",
            "size": 1
        }

        response = requests.get(url_gdc_cases, params=params)
        data = response.json()

        hits = data.get("data", {}).get("hits", [])
        if not hits:
            raise ValueError(f"No case found for barcode {barcode}")
    except Exception as e:
        print(f"No data found for {barcode}. error: {e}")
        hits = {0: {'case_id': barcode}}

    return hits[0]["case_id"]

def get_files(case_uuid, data_type="Gene Expression Quantification"):
    

    try:
        filters = {
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "cases.case_id",
                        "value": [case_uuid]
                    }
                },
                {
                    "op": "in",
                    "content": {
                        "field": "data_type",
                        "value": [data_type]
                    }
                }
            ]
        }

        params = {
            "filters": json.dumps(filters),
            "fields": "file_id,file_name,data_type",
            "format": "JSON",
            "size": 100
        }

        response = requests.get(url_gdc_files, params=params)
        data = response.json()

    except Exception as e:
        print(f"No data found for {case_uuid}. error: {e}")
        return None

    return data["data"]["hits"]

def download_file(file_id):

    fname = f"{file_id}.dat"
    ret = True

    try:
        os.makedirs(root_case, exist_ok=True)

        url = url_gdc_data%(file_id)
        response = requests.get(url, stream=True)

        filename = os.path.join(root_case, fname)

        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    except Exception as e:
        print(f"No data found for {file_id}. error: {e}")
        ret = False
           
    return ret, fname

def clean_gdc_tsv(filename, value_col="tpm_unstranded") -> pd.DataFrame:
    # Load file (auto-detect comments)

    out_filename = filename.replace(".dat", ".tsv")

    if os.path.exists(out_filename):
        df2 = pd.read_csv(out_filename, sep='\t')
        print(f"Reading table {df2.shape}: {out_filename}")
        return df2
    
    try:
        df = pd.read_csv(filename, sep="\t", comment="#")

        # Remove summary rows (N_*)
        df = df[~df["gene_id"].str.startswith("N_")]

        # Keep only valid Ensembl genes
        df = df[df["gene_id"].str.startswith("ENSG")]

        # Remove version from gene_id (ENSG... -> ENSG...)
        df["gene_id"] = df["gene_id"].str.split(".").str[0]

        # Optional: keep only relevant columns
        cols = ["gene_id", "gene_name", "gene_type", value_col]
        df2 = df[cols].copy()

        # Rename for clarity
        df2 = df2.rename(columns={"gene_name": "symbol", value_col: "expression"})

        df2.to_csv(out_filename, sep="\t", index=False)
        print(f"Read table {df2.shape}: {out_filename}")
    except Exception as e:
        print(f"Could not prepare and save '{out_filename}' - error: {e}")
        return pd.DataFrame()
    
    return df2

### Barcode: case

In [ ]:
case_uuid2 = 'e3935ce4-64d3-4a66-ba11-d308b844b410'
barcode = 'TCGA-E9-A5FL'

case_uuid = get_case_uuid(barcode)

case_uuid, case_uuid2

### Get gene expression

In [ ]:
response = get_files(case_uuid, data_type="Gene Expression Quantification")

type(response), len(response)

In [ ]:
type(response[0]), response[0]

In [ ]:
try:
    dic = response[0]
    if isinstance(dic, str):
        dic = eval(dic)

    file_name = dic['file_name']
    file_id = dic['file_id']
    data_type = dic['data_type']

except Exception as e:
    print(f"No response. error: {e}")

    file_name = ''
    file_id = ''
    data_type = ''

print(f"file_name '{file_name} \nfile_id '{file_id} \ndata_type '{data_type}'")

In [ ]:
ret, fname = download_file(file_id)

if ret:
    filename = os.path.join(root_case, fname)
    print(os.path.join(root_case, filename))
else:
    filename = 'not_found.tsv'

filename

In [ ]:
filename = os.path.join(root_case, fname)
df2 = clean_gdc_tsv(filename, value_col="tpm_unstranded")

df2.head(3)